# 03 – HMM sur les rendements réels + prédiction du prochain régime
Données : bougies de **8 h** (3 par jour). Observations : rendement log et volatilité. Étapes : features → découpage train/test → choix de N (AIC/BIC) → interprétation → **prédiction sans look-ahead bias**.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler
np.set_printoptions(precision=4, suppress=True)

FICHIER   = "../data/btcusdt_8h.csv"   # bougies de 8h (3 par jour)
FENETRE   = 60                         # volatilité : 20 jours x 3 bougies
PAR_JOUR  = 3                          # bougies par jour

ModuleNotFoundError: No module named 'hmmlearn'

## 1. Features et découpage chronologique

In [ ]:
df = pd.read_csv(FICHIER, index_col=0, parse_dates=True).sort_index()
df["rendement"]  = np.log(df["Close"] / df["Close"].shift(1))
df["volatilite"] = df["rendement"].rolling(FENETRE).std()
df = df.dropna()

colonnes = ["rendement", "volatilite"]
coupure  = int(len(df) * 0.7)                       # 70 % train / 30 % test, dans l'ordre du temps
train, test = df.iloc[:coupure], df.iloc[coupure:]

scaler  = StandardScaler().fit(train[colonnes])     # fit UNIQUEMENT sur le train
X_train = scaler.transform(train[colonnes])
X_test  = scaler.transform(test[colonnes])
print(f"{len(df)} lignes | train : {train.index[0].date()} -> {train.index[-1].date()} ({len(train)}) | test : {test.index[0].date()} -> {test.index[-1].date()} ({len(test)})")

## 2. Choix du nombre d'états N (AIC / BIC)
Nombre de paramètres d'un HMM gaussien à covariance pleine, avec `d` variables : `k = (N−1) + N(N−1) + N·d + N·d(d+1)/2`.

In [ ]:
def meilleur_modele(X, N, n_init=8):
    best, best_ll = None, -np.inf
    for seed in range(n_init):                      # plusieurs initialisations (optimum local)
        m = GaussianHMM(n_components=N, covariance_type="full", n_iter=300, tol=1e-4, random_state=seed)
        try:
            m.fit(X); ll = m.score(X)
        except Exception:
            continue
        if ll > best_ll: best, best_ll = m, ll
    return best, best_ll

d, n = X_train.shape[1], len(X_train)
lignes, modeles = [], {}
for N in range(2, 6):
    m, ll = meilleur_modele(X_train, N)
    k = (N-1) + N*(N-1) + N*d + N*d*(d+1)//2
    lignes.append({"N": N, "logL": ll, "k": k, "AIC": 2*k - 2*ll, "BIC": k*np.log(n) - 2*ll})
    modeles[N] = m
tab = pd.DataFrame(lignes).set_index("N"); display(tab.round(1))
N_BIC = int(tab["BIC"].idxmin()); N_AIC = int(tab["AIC"].idxmin())
print("N optimal selon BIC :", N_BIC, "| selon AIC :", N_AIC)
tab[["AIC", "BIC"]].plot(marker="o", figsize=(6, 3.5), title="AIC / BIC selon N"); plt.show()

## 3. Modèle retenu et interprétation économique
Le sujet demande 3 états (haussier, baissier, panique). On prend `N = 3` pour l'interprétation et on compare avec le choix du BIC dans le rapport. Les états sont **triés par rendement moyen** pour que « 0 = le plus baissier ».

In [ ]:
N = 3
model = modeles[N]
ordre = np.argsort(model.means_[:, 0])               # tri par rendement moyen croissant
mu_brut = scaler.inverse_transform(model.means_)[ordre]
A = model.transmat_[np.ix_(ordre, ordre)]
pi0 = model.startprob_[ordre]
covs = model.covars_[ordre]; means = model.means_[ordre]

resume = pd.DataFrame(mu_brut, columns=["rendement moyen / bougie", "volatilité moyenne"])
resume["durée moyenne (bougies)"] = 1 / (1 - np.diag(A))
resume["durée moyenne (jours)"]  = resume["durée moyenne (bougies)"] / PAR_JOUR
resume.index = ["État 0 (rendement le plus bas)", "État 1", "État 2 (rendement le plus haut)"]
display(resume.round(5))
print("Matrice de transition (états triés) :\n", A.round(3))

In [ ]:
# Étiquetage automatique : rendement le plus haut = Haussier ; volatilité la plus forte = Panique ; reste = Baissier
vol = mu_brut[:, 1]
etiquettes = {}
etiquettes[int(np.argmax(vol))] = "Panique"
reste = [i for i in range(3) if i not in etiquettes]
etiquettes[reste[int(np.argmax(mu_brut[reste, 0]))]] = "Haussier"
etiquettes[[i for i in range(3) if i not in etiquettes][0]] = "Baissier"
NOMS = [etiquettes[i] for i in range(3)]
print("Étiquettes :", dict(enumerate(NOMS)), "(à vérifier à la main dans le rapport)")

## 4. Prédiction **sans look-ahead bias** (filtrage forward)
`predict_proba` de hmmlearn utilise passé **et futur** de la série (lissage). Pour le trading, il faut le **filtre forward** : `P(S_t | x_1..x_t)`. Puis la prédiction : `P(S_{t+1}) = filtre_t · A`.

In [ ]:
def filtre_forward(X, pi0, A, means, covs):
    T, N = len(X), len(pi0)
    emis = np.column_stack([multivariate_normal(means[i], covs[i], allow_singular=True).pdf(X) for i in range(N)])
    F = np.zeros((T, N))
    f = pi0 * emis[0]; F[0] = f / f.sum()
    for t in range(1, T):
        f = (F[t-1] @ A) * emis[t]; F[t] = f / f.sum()
    return F

X_tout  = np.vstack([X_train, X_test])
F       = filtre_forward(X_tout, pi0, A, means, covs)   # modèle appris sur le train seulement
F_test  = F[len(X_train):]
P_next  = F_test @ A                                    # probabilité du régime à t+1
regime_pred = P_next.argmax(axis=1)

res = test.copy()
res["regime_actuel"]  = [NOMS[i] for i in F_test.argmax(axis=1)]
res["regime_predit"]  = [NOMS[i] for i in regime_pred]
res["rendement_suivant"] = res["rendement"].shift(-1)
res = res.iloc[:-1]
display(res[["rendement", "regime_actuel", "regime_predit"]].tail(6))

In [ ]:
# Évaluation : le régime prédit renseigne-t-il sur le rendement de la bougie suivante ?
stats = res.groupby("regime_predit")["rendement_suivant"].agg(["count", "mean", "std"])
stats["rendement annualisé"] = stats["mean"] * PAR_JOUR * 365
display(stats.round(5))

# Comparaison avec la méthode naïve « le régime de demain = celui d'aujourd'hui »
naif = (res["regime_actuel"] == res["regime_predit"]).mean()
print(f"Le régime prédit = le régime actuel dans {naif:.1%} des cas (persistance des régimes)")

**Lecture.** Si ce pourcentage est proche de 100 %, c'est normal : la diagonale de A est très grande (régimes persistants), donc le régime le plus probable demain est presque toujours celui d'aujourd'hui. L'apport du HMM n'est pas de changer le régime prédit, mais de donner la **probabilité de changer de régime** : `P(changement) = 1 − P(S_{t+1} = régime actuel)`. On vérifie qu'elle est bien calibrée.

In [ ]:
idx_act = F_test.argmax(axis=1)[:-1]
p_chg = 1 - P_next[:-1][np.arange(len(idx_act)), idx_act]          # probabilité prédite de changer de régime
chg_reel = (F_test.argmax(axis=1)[1:] != idx_act).astype(int)      # changement observé à t+1
calib = pd.DataFrame({"p_chg": p_chg, "reel": chg_reel})
calib["classe"] = pd.qcut(calib["p_chg"], 4, duplicates="drop")
display(calib.groupby("classe", observed=True).agg(proba_predite=("p_chg", "mean"), frequence_reelle=("reel", "mean"), n=("reel", "size")).round(4))

In [ ]:
# Graphique : prix colorié par régime (test)
couleurs = {"Haussier": "green", "Baissier": "red", "Panique": "orange"}
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(res.index, res["Close"], color="lightgrey", lw=0.8)
for nom, col in couleurs.items():
    sel = res["regime_actuel"] == nom
    ax.scatter(res.index[sel], res["Close"][sel], s=3, color=col, label=nom)
ax.set_yscale("log"); ax.legend(markerscale=4); ax.set_title("Régimes filtrés (sans look-ahead) sur la période de test")
plt.tight_layout(); plt.savefig("../results/regimes_test.png", dpi=150); plt.show()